# Seeing-aware effective source density n_eff on the dust-footprint variants

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-21
- Kernel: conda_py313_opsim53
- Context: series `10_DESCMAFDEPTHANDGALCOUNT`. Notebook 03 gives the weak-lensing proxy metrics of `rubin_sim` (number of visits, exposure time), which contain no seeing. Notebook 04 built and tested a forward model of the weak-lensing effective source density `n_eff` as a function of seeing and depth (Chang et al. 2013). **This notebook applies that model, pixel by pixel, to the 7 simulations**: it is the first prototype of a seeing-aware weak-lensing MAF metric.
- This notebook: **05** of the series. Companion notebooks: `01_compareExgalM5withCuts.ipynb`, `03_WeakLensing.ipynb` (footprint), `04_NeffSeeingModel.ipynb` (model, calibration on HSC / KiDS / DES).
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`): the 7 runs of the previous notebooks (`shrink_fp_dust_{0.050, 0.080, 0.120, 0.150, 0.199, 0.250}_v5.3.6_10yrs.db` and `baseline_v5.3.6_11yrs.db`, E(B-V) threshold 0.200).

## Notebook overview

**Pipeline, for each run and each pixel (`nside = 64`).**
1. **Footprint**: the WL footprint of notebook 03 (pixels where `WeakLensingNvisits` in `gri` is defined: dust cut adapted to the run, `i`-band coadded depth >= 25.9, minimum exposure time). It is read from the cache of notebook 03, which must have been run.
2. **Depth**: the dust-corrected coadded 5-sigma depth in `r` and in `i` (`ExgalM5WithCuts` with all cuts disabled, one band at a time), computed with MAF here.
3. **Seeing**: the effective PSF of the coadd in `r` and in `i`, `sqrt(mean(FWHM^2))` over the visits of the pixel (mean of `r_PSF^2`, as in Chang et al. 2013), computed with a small custom MAF metric. The column is `seeingFwhmGeom` by default (see below).
4. **Model**: `n_eff` from the joint `r + i` fit of notebook 04, vectorized over the pixels, in two variants: `generic` (cut `sigma_m < sigma_SN`) and `R2cut` (same plus the resolution cut `R2 > 0.3` of HSC, applied to the `i` band).
5. **Totals**: `N_eff = sum_pixels n_eff * pixel_area` (effective number of galaxies of the footprint), and `sqrt(N_eff)` relative to the baseline as the statistical signal-to-noise proxy in the shape-noise-dominated regime.

**Choices (editable in Section 2).**
- `SEEING_COL = 'seeingFwhmGeom'`: geometric FWHM, the quantity comparable to the seeing quoted by HSC, KiDS and DES (used in notebook 04); the depth already contains the effective seeing through `fiveSigmaDepth`. `'seeingFwhmEff'` can be used instead (larger by about 10-20%).
- Population and colours are the ASSUMED ones of notebook 04 (`POP`, `OFFSET`), deblending radius 2 arcsec, `sigma_SN = 0.26`. The absolute `n_eff` carries the modelling uncertainty quantified in notebook 04 (published / model between 0.65 and 1.4 for the three surveys): use the **relative** comparison between runs.
- Same first-10-years truncation, non-DDF visits and per-run E(B-V) cut as in notebooks 01-03 (`EBV_CUT_MODE = 'run'`).

**Outputs**: MAF results and cached maps in `data_05_NEFFMAPS/`, figures in `figs_05_NEFFMAPS/`. MAF is run once per (run, band, quantity); the `n_eff` maps are cached with a tag that contains the model choices.

## 1. Imports

In [ ]:
import os
import inspect
from glob import glob
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maf_maps
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.metrics import BaseMetric
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import ExgalM5WithCuts

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

In [ ]:
NB_TAG = "NEFFMAPS"
data_dir = f"data_05_{NB_TAG}"
figs_dir = f"figs_05_{NB_TAG}"
DIR01 = "data_01_EXGALM5CUTS"  # caches of notebook 01 (used for a consistency check)
DIR03 = "data_03_WL"  # caches of notebook 03 (WL footprint)
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

OPSIM_DIR = "/Users/dagoret/DATA/OpSim"
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_11yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_11yrs"
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]


def get_db_path(run_name):
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


NSIDE = 64
MAX_NIGHT = 10 * 365.25 + 0.5  # first 10 years of every run
BANDS = ("r", "i")  # bands of the joint shape measurement
SHAPE_BAND = "i"  # band on which the resolution cut is applied
SEEING_COL = "seeing" + "FwhmGeom"  # geometric FWHM; use seeingFwhmEff for the effective one
EBV_CUT_MODE = "run"  # must be the mode used to build the caches of notebook 03
FIXED_LIM_EBV = 0.2
FORCE_RECOMPUTE = False  # True -> redo the MAF runs
FORCE_NEFF = False  # True -> recompute the n_eff maps

# --- model (Chang et al. 2013), see notebook 04
SIGMA_SN = 0.26
ABC = (1.58, 5.03, 0.39)
BLEND = {1.0: (0.50, 1.9e-3), 2.0: (0.68, 5.5e-3), 3.0: (0.62, 1.4e-2)}
BLEND_D = 2.0
POP = dict(rh0=0.35, xi=0.13, sig_lnr=0.45, size_scale=1.0)  # ASSUMED galaxy population, as in notebook 04
OFFSET = dict(r=0.4, i=0.0, z=-0.1)  # ASSUMED colours m_band = m_i + OFFSET[band]
R2_MIN = 0.3  # HSC-like resolution cut R2 = R / (1 + R) > R2_MIN
VARIANTS = {"generic": None, "R2cut": R2_MIN / (1.0 - R2_MIN)}  # variant -> minimum R on the shape band
POP_TAG = "rh%.2f_xi%.2f_sl%.2f_ss%.2f" % (POP["rh0"], POP["xi"], POP["sig_lnr"], POP["size_scale"])

pix_area = hp.nside2pixarea(NSIDE, degrees=True)  # deg^2
pix_arcmin2 = pix_area * 3600.0


def metric_ebv_cut(run_name):
    if EBV_CUT_MODE == "run":
        return DUST_THRESH[run_name]
    if EBV_CUT_MODE == "fixed":
        return FIXED_LIM_EBV
    raise ValueError(f"EBV_CUT_MODE must be run or fixed, got {EBV_CUT_MODE!r}")


def sql_band(band):
    return (
        "scheduler_note not like "
        + repr("DD%")
        + " and band="
        + repr(band)
        + " and night <= "
        + repr(MAX_NIGHT)
    )


print(
    f"nside={NSIDE}, pixel area = {pix_area:.4f} deg^2 = {pix_arcmin2:.1f} arcmin^2; seeing column: {SEEING_COL}"
)

## 3. The model, vectorized over the pixels

Same formulas as `neff()` of notebook 04 (Chang et al. 2013): `nu`, `R` and `sigma_m` per band, joint fit over the bands, selection `sigma_m < sigma_SN` (and optionally `R >= R_min` on the shape band), weights `sigma_SN^2 / (sigma_SN^2 + sigma_m^2)`, blending. A band with no visit in a pixel (non-finite depth or seeing) is given a negligible weight, so that the pixel is still computed from the other band; the shape band must be valid.

In [ ]:
def dndm_i(m):
    # differential counts [arcmin^-2 mag^-1] from N(<i) = 46 * 10**(0.31 * (i - 25))  (LSST Science Book)
    return 46.0 * 0.31 * np.log(10.0) * 10 ** (0.31 * (m - 25.0))


def sigma_m(nu, R, abc=ABC):
    a, b, c = abc
    return a / nu * (1.0 + (b / R) ** c)


def neff_map(
    m5,
    fwhm,
    bands=BANDS,
    shape_band=SHAPE_BAND,
    k=1.0,
    R_min=None,
    blend_d=BLEND_D,
    pop=POP,
    dm=0.05,
    nz=41,
    chunk=200,
):
    # m5, fwhm: dict band -> 1-D array over the pixels (NaN = no visit). Returns n_raw and n_eff [arcmin^-2] (NaN where undefined).
    m_i = np.arange(20.0, 28.5, dm) + dm / 2
    z = np.linspace(-3.5, 3.5, nz)
    wz = np.exp(-0.5 * z**2)
    wz /= wz.sum()
    rh = pop["size_scale"] * pop["rh0"] * 10 ** (-pop["xi"] * (m_i - 24.0))
    r_gal = (
        1.46 * rh[:, None] * np.exp(pop["sig_lnr"] * z[None, :])
    )  # (nmag, nz), second-moment radius of a disk
    dN = (dndm_i(m_i) * dm)[:, None] * wz[None, :]
    npix = len(m5[shape_band])
    ok = np.isfinite(m5[shape_band]) & np.isfinite(fwhm[shape_band]) & (fwhm[shape_band] > 0)
    m5v, fwv = {}, {}
    for b in bands:  # missing band -> hopelessly shallow, zero weight in the joint fit
        good = np.isfinite(m5[b]) & np.isfinite(fwhm[b]) & (fwhm[b] > 0)
        m5v[b] = np.where(good, m5[b], 0.0)
        fwv[b] = np.where(good, fwhm[b], 1.0)
    n_raw = np.full(npix, np.nan)
    n_eff = np.full(npix, np.nan)
    idx = np.flatnonzero(ok)
    for s in range(0, len(idx), chunk):
        ii = idx[s : s + chunk]
        inv = 0.0
        for b in bands:
            sig = (fwv[b][ii] / 2.3548)[:, None, None]
            r_psf = np.sqrt(2.0) * sig
            r_ap = 1.64 * np.sqrt(r_gal[None] ** 2 + r_psf**2)
            nu_ps = 5.0 * 10 ** (0.4 * (m5v[b][ii][:, None, None] - (m_i + OFFSET[b])[None, :, None]))
            nu = 0.9 * nu_ps * 2.0 * sig / r_ap
            R = r_gal[None] ** 2 / r_psf**2
            inv = inv + 1.0 / sigma_m(nu, R) ** 2
            if b == shape_band:
                R_shape = R
        sm = 1.0 / np.sqrt(inv)
        sel = sm < k * SIGMA_SN
        if R_min is not None:
            sel = sel & (R_shape >= R_min)
        raw = (dN[None] * sel).sum(axis=(1, 2))
        n0 = (dN[None] * sel * SIGMA_SN**2 / (SIGMA_SN**2 + sm**2)).sum(axis=(1, 2))
        fb = 0.0
        if blend_d is not None:
            eta, mu = BLEND[blend_d]
            fb = eta * np.log(1.0 + mu * raw)
        n_raw[ii] = raw
        n_eff[ii] = n0 * (1.0 - fb)
    return n_raw, n_eff

**Validation against notebook 04**: an LSST-like pixel (10-year coadd depths `r = 27.5`, `i = 26.8`, seeing 0.8 arcsec in both bands) must give the values printed in Section 7 of notebook 04, `n_eff = 36.1` (generic) and `31.9` (with the resolution cut applied on the `r` band, which is the band used for the cut in that notebook).

In [ ]:
one = lambda v: np.array([v])
m5_test = dict(r=one(27.5), i=one(26.8))
fw_test = dict(r=one(0.8), i=one(0.8))
gen = neff_map(m5_test, fw_test)[1][0]
cut = neff_map(m5_test, fw_test, shape_band="r", R_min=R2_MIN / (1.0 - R2_MIN))[1][0]
print(f"generic: {gen:.2f} (notebook 04: 36.1)   R2 > 0.3 on r: {cut:.2f} (notebook 04: 31.9)")

## 4. Coadded depth and effective seeing per pixel (MAF)

For each run and each band of `BANDS`:
- `M5_<band>`: `ExgalM5WithCuts(lsst_filter=<band>)` with every cut disabled (`extinction_cut = 99`, `n_filters = 1`, `depth_cut = 0`) on the visits of that band only: the dust-corrected coadded 5-sigma depth;
- `FWHM_<band>`: `sqrt(mean(FWHM^2))` of the visits of that band (custom metric below).

The signature of `ExgalM5WithCuts` is printed for reference.

In [ ]:
print("ExgalM5WithCuts", inspect.signature(ExgalM5WithCuts.__init__))


class CoaddFwhmMetric(BaseMetric):
    # Effective PSF of the coadd of the visits of a pixel: sqrt(mean(FWHM^2)), i.e. mean of r_PSF^2 (Chang et al. 2013)
    def __init__(self, seeing_col=SEEING_COL, metric_name="CoaddFwhm"):
        self.seeing_col = seeing_col
        super().__init__(col=[seeing_col], metric_name=metric_name, units="arcsec")

    def run(self, data_slice, slice_point=None):
        return float(np.sqrt(np.mean(np.asarray(data_slice[self.seeing_col], dtype=float) ** 2)))

In [ ]:
dustmap = maf_maps.DustMap(nside=NSIDE, interp=False)


def bundle_to_masked(bundle):
    data = np.ma.getdata(bundle.metric_values).astype(float)
    mask = np.ma.getmaskarray(bundle.metric_values) | ~np.isfinite(data) | (data < -600.0)
    return np.ma.masked_array(data, mask=mask)


def save_map(path, m):
    np.savez_compressed(path, data=np.ma.getdata(m), mask=np.ma.getmaskarray(m))


def load_map(path):
    with np.load(path) as f:
        return np.ma.masked_array(f["data"], mask=f["mask"])


def cache_path(quantity, band, run_name):
    tag = run_name.replace(".", "_")
    if quantity == "M5":
        return join(data_dir, f"M5_{tag}_{band}_nside{NSIDE}.npz")
    return join(data_dir, f"FWHM_{SEEING_COL}_{tag}_{band}_nside{NSIDE}.npz")


def load_or_run(run_name):
    keys = [(q, b) for q in ("M5", "FWHM") for b in BANDS]
    paths = {key: cache_path(key[0], key[1], run_name) for key in keys}
    todo = [key for key in keys if FORCE_RECOMPUTE or not isfile(paths[key])]
    out = {key: load_map(paths[key]) for key in keys if key not in todo}
    if not todo:
        print(f"[{run_name}] loaded cached depth and seeing maps")
        return out
    dbpath = get_db_path(run_name)
    print(f"[{run_name}] computing {todo} on {dbpath}")
    slicer = slicers.HealpixSlicer(nside=NSIDE, use_cache=False)
    bundles = {}
    for quantity, band in todo:
        if quantity == "M5":
            metric = ExgalM5WithCuts(
                lsst_filter=band,
                extinction_cut=99.0,
                n_filters=1,
                depth_cut=0.0,
                metric_name="ExgalM5_" + band,
            )
            bundles[(quantity, band)] = mb.MetricBundle(
                metric,
                slicer,
                sql_band(band),
                maps_list=[dustmap],
                run_name=run_name,
                info_label="M5 " + band + " 10yr nonDD",
            )
        else:
            metric = CoaddFwhmMetric(metric_name="CoaddFwhm_" + band)
            bundles[(quantity, band)] = mb.MetricBundle(
                metric,
                slicer,
                sql_band(band),
                run_name=run_name,
                info_label=SEEING_COL + " " + band + " 10yr nonDD",
            )
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list(list(bundles.values())), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()
    for key, bundle in bundles.items():
        out[key] = bundle_to_masked(bundle)
        save_map(paths[key], out[key])
    return out


maf_maps_by_run = {run_name: load_or_run(run_name) for run_name in RUN_NAMES}

## 5. Consistency check with notebook 01

The `i`-band depth computed here (all cuts disabled) must be identical to the `ExgalM5WithCuts` map of notebook 01 in the pixels where the latter is defined (same coadd, same dust correction). A non-zero difference would mean that the permissive call above does not do what is expected.

In [ ]:
def load_nb01_depth(run_name):
    tag = run_name.replace(".", "_")
    ebv = metric_ebv_cut(run_name)
    hits = glob(join(DIR01, f"ExgalM5WithCuts_{tag}_i_nf6_ebv{ebv:.3f}_*nside{NSIDE}.npz"))
    return load_map(hits[0]) if hits else None


rows = []
for run_name in RUN_NAMES:
    ref = load_nb01_depth(run_name)
    mine = maf_maps_by_run[run_name][("M5", "i")]
    if ref is None:
        rows.append(dict(run=run_name, n_common=np.nan, max_abs_diff=np.nan))
        continue
    both = ~np.ma.getmaskarray(ref) & ~np.ma.getmaskarray(mine)
    d = np.abs(np.ma.getdata(ref)[both] - np.ma.getdata(mine)[both])
    rows.append(
        dict(
            run=run_name,
            n_common=int(both.sum()),
            max_abs_diff=d.max() if d.size else np.nan,
            n_valid_nb01_only=int((~np.ma.getmaskarray(ref) & np.ma.getmaskarray(mine)).sum()),
        )
    )
display(pd.DataFrame(rows).set_index("run"))

## 6. Inputs of the model on the WL footprint

The footprint is the set of pixels where `WeakLensingNvisits` (`gri`) is defined in notebook 03 for the same E(B-V) cut. Outside it, the inputs are set to NaN.

In [ ]:
def load_footprint(run_name):
    tag = run_name.replace(".", "_")
    ebv = metric_ebv_cut(run_name)
    hits = glob(join(DIR03, f"NV_gri_{tag}_*ebv{ebv:.3f}_nside{NSIDE}.npz"))
    if not hits:
        raise FileNotFoundError(
            f"No WL cache in {DIR03} for {run_name} (E(B-V) cut {ebv:.3f}): run notebook 03 first"
        )
    nv = load_map(hits[0])
    return ~np.ma.getmaskarray(nv), nv


inputs = {}
rows = []
for run_name in RUN_NAMES:
    foot, nv = load_footprint(run_name)
    m5 = {b: np.where(foot, maf_maps_by_run[run_name][("M5", b)].filled(np.nan), np.nan) for b in BANDS}
    fw = {b: np.where(foot, maf_maps_by_run[run_name][("FWHM", b)].filled(np.nan), np.nan) for b in BANDS}
    inputs[run_name] = dict(foot=foot, nv=nv, m5=m5, fwhm=fw)
    row = {"run": run_name, "footprint area [deg2]": foot.sum() * pix_area}
    for b in BANDS:
        row[f"median m5 {b}"] = np.nanmedian(m5[b])
        row[f"median FWHM {b} [arcsec]"] = np.nanmedian(fw[b])
        row[f"{b}: pixels without visits"] = int((foot & ~np.isfinite(m5[b])).sum())
    rows.append(row)
inputs_summary = pd.DataFrame(rows).set_index("run")
inputs_summary.to_csv(join(data_dir, "neff_inputs_summary.csv"))
display(inputs_summary.round(2))

## 7. n_eff maps

Two variants are computed for each run: `generic` and `R2cut`. The maps are cached; the tag contains the seeing column, the population parameters, the deblending radius and the bands.

In [ ]:
def neff_cache_path(variant, run_name):
    tag = run_name.replace(".", "_")
    return join(
        data_dir,
        f'NEFF_{variant}_{tag}_{SEEING_COL}_{"".join(BANDS)}_{POP_TAG}_blend{BLEND_D:.0f}_nside{NSIDE}.npz',
    )


def get_neff(variant, run_name):
    path = neff_cache_path(variant, run_name)
    if isfile(path) and not FORCE_NEFF:
        return load_map(path)
    inp = inputs[run_name]
    _, n_eff = neff_map(inp["m5"], inp["fwhm"], R_min=VARIANTS[variant])
    m = np.ma.masked_invalid(n_eff)
    save_map(path, m)
    return m


neff_maps = {}
for variant in VARIANTS:
    neff_maps[variant] = {}
    for run_name in RUN_NAMES:
        neff_maps[variant][run_name] = get_neff(variant, run_name)
        print(f"[{variant}] {run_name}: {neff_maps[variant][run_name].count()} pixels")

In [ ]:
def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def as_healpy(m):
    return np.ma.filled(m, hp.UNSEEN)


def per_run_summary(maps):
    rows = []
    for run_name, dust in RUNS_INFO:
        v = maps[run_name].compressed()
        rows.append(
            {
                "run": run_name,
                "E(B-V) cut": dust,
                "area_deg2": v.size * pix_area,
                "mean n_eff": v.mean() if v.size else np.nan,
                "median n_eff": np.median(v) if v.size else np.nan,
                "std n_eff": v.std() if v.size else np.nan,
                "N_eff [1e6]": v.sum() * pix_arcmin2 / 1e6,
            }
        )
    df = pd.DataFrame(rows).set_index("run")
    ref = df.loc[REF_RUN, "N_eff [1e6]"]
    df["N_eff / baseline"] = df["N_eff [1e6]"] / ref
    df["sqrt(N_eff) / baseline"] = np.sqrt(df["N_eff / baseline"])
    return df


summaries = {}
for variant in VARIANTS:
    summaries[variant] = per_run_summary(neff_maps[variant])
    summaries[variant].to_csv(join(data_dir, f"neff_{variant}_per_run_summary.csv"))
    print(f"n_eff [arcmin^-2], variant {variant}:")
    display(summaries[variant].round(3))

## 8. Healpix maps and histograms

In [ ]:
def plot_all_maps(maps, tag, title, unit="n_eff [arcmin$^{-2}$]", cmap="viridis"):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    if pooled.size == 0:
        print(f"{title}: all pixels are masked, nothing to plot")
        return
    vmin, vmax = np.percentile(pooled, 1), np.percentile(pooled, 99)
    fig = plt.figure(figsize=(15, 12))
    for i, (run_name, dust) in enumerate(RUNS_INFO, start=1):
        hp.mollview(
            as_healpy(maps[run_name]),
            fig=fig.number,
            sub=(3, 3, i),
            min=vmin,
            max=vmax,
            cmap=cmap,
            title=f"{run_name}\n{run_label(run_name)}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(f"{title} (common color scale: 1st-99th percentile of all runs)", fontsize=16, y=1.02)
    save_fig(fig, f"{tag}_healpix_allruns")
    plt.show()


def plot_overlay_hist(maps, tag, title, xlabel="n_eff [arcmin$^{-2}$]", nbins=80):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    bins = np.linspace(np.percentile(pooled, 0.1), np.percentile(pooled, 99.9), nbins + 1)
    colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    for (run_name, dust), c in zip(RUNS_INFO, colors):
        v = maps[run_name].compressed()
        w = np.full(v.size, pix_area)
        axs[0].hist(v, bins=bins, weights=w, histtype="step", color=c, label=run_label(run_name))
        axs[1].hist(
            v, bins=bins, weights=w, histtype="step", color=c, cumulative=-1, label=run_label(run_name)
        )
    axs[0].set_ylabel("Area per bin [deg²]")
    axs[1].set_ylabel("Area with value ≥ x [deg²]")
    for ax in axs:
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.3)
    axs[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, f"{tag}_histograms_allruns")
    plt.show()


for variant in VARIANTS:
    plot_all_maps(neff_maps[variant], f"NEFF_{variant}", f"n_eff, variant {variant}")
    plot_overlay_hist(
        neff_maps[variant],
        f"NEFF_{variant}",
        f"n_eff, variant {variant}: area-weighted histograms (left), area with value >= x (right)",
    )

## 9. Differences between consecutive dust thresholds

As in notebooks 02 and 03, a masked pixel counts as 0, so the maps contain the pixels gained or lost with the footprint. The table gives, in effective galaxies (`N_eff`, in millions), the change of the total split into the change on the common pixels, the pixels gained and the pixels lost. Variant `generic` only (the same code applies to `R2cut`).

In [ ]:
def diff_map(map_b, map_a):
    mask_b, mask_a = np.ma.getmaskarray(map_b), np.ma.getmaskarray(map_a)
    b = np.where(mask_b, 0.0, np.ma.getdata(map_b).astype(float))
    a = np.where(mask_a, 0.0, np.ma.getdata(map_a).astype(float))
    return np.ma.masked_array(b - a, mask=mask_b & mask_a)


def pair_table(maps):
    rows = []
    for run_b, run_a in PAIRS:
        mb_, ma_ = maps[run_b], maps[run_a]
        vb, va = ~np.ma.getmaskarray(mb_), ~np.ma.getmaskarray(ma_)
        b = np.ma.getdata(mb_).astype(float)
        a = np.ma.getdata(ma_).astype(float)
        common, gained, lost = vb & va, vb & ~va, va & ~vb
        f = pix_arcmin2 / 1e6
        rows.append(
            {
                "pair (E(B-V) thresholds)": f"{DUST_THRESH[run_b]:.3f} - {DUST_THRESH[run_a]:.3f}",
                "area gained [deg2]": gained.sum() * pix_area,
                "area lost [deg2]": lost.sum() * pix_area,
                "dN_eff total [1e6]": (b[vb].sum() - a[va].sum()) * f,
                "from common pixels": (b[common] - a[common]).sum() * f,
                "from gained pixels": b[gained].sum() * f,
                "from lost pixels": -a[lost].sum() * f,
            }
        )
    return pd.DataFrame(rows).set_index("pair (E(B-V) thresholds)")


def plot_diff_mosaic(maps, tag, title):
    diffs = {pair: diff_map(maps[pair[0]], maps[pair[1]]) for pair in PAIRS}
    lims = [np.percentile(np.abs(d.compressed()), 99) for d in diffs.values() if d.count()]
    vlim = max(max(lims), 1e-6) if lims else 1.0
    fig = plt.figure(figsize=(16, 10))
    for i, ((run_b, run_a), d) in enumerate(diffs.items(), start=1):
        hp.mollview(
            as_healpy(d),
            fig=fig.number,
            sub=(2, 3, i),
            min=-vlim,
            max=vlim,
            cmap="RdBu_r",
            title=f"{run_b}\nminus {run_a}",
            unit="Δ n_eff [arcmin$^{-2}$]",
            cbar=True,
        )
    fig.suptitle(
        f"{title}: differences between consecutive dust thresholds (common scale ±{vlim:.3g})",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_diff_allpairs_combined")
    plt.show()


plot_diff_mosaic(neff_maps["generic"], "NEFF_generic", "n_eff (generic)")
pairs_generic = pair_table(neff_maps["generic"])
pairs_generic.to_csv(join(data_dir, "neff_generic_diff_summary.csv"))
display(pairs_generic.round(3))

## 10. N_eff and n_eff versus the dust threshold

Left: total effective number of galaxies of the footprint (`N_eff`, millions), proportional to the statistical weight for a shape-noise-dominated measurement. Right: mean and median `n_eff` over the footprint; the light band is the mean ± the pixel-to-pixel standard deviation (variant `generic`; the `R2cut` variant is drawn with thin lines).

In [ ]:
x = np.array([DUST_THRESH[r] for r in RUN_NAMES])
i_ref = RUN_NAMES.index(REF_RUN)
fig, axs = plt.subplots(1, 2, figsize=(14, 5.5))
styles = {"generic": ("tab:blue", "o"), "R2cut": ("tab:red", "s")}
for variant, (color, marker) in styles.items():
    s = summaries[variant]
    axs[0].plot(x, s["N_eff [1e6]"].values, marker + "-", color=color, label=variant)
axs[0].plot(
    x[i_ref],
    summaries["generic"]["N_eff [1e6]"].values[i_ref],
    "o",
    mfc="none",
    mec="k",
    ms=12,
    label="baseline",
)
axs[0].set_xlabel("E(B-V) threshold that defines the footprint of the run")
axs[0].set_ylabel("N_eff [millions of galaxies]")
axs[0].grid(alpha=0.3)
axs[0].legend(fontsize=8)

s = summaries["generic"]
mean, median, std = s["mean n_eff"].values, s["median n_eff"].values, s["std n_eff"].values
axs[1].fill_between(
    x, mean - std, mean + std, color="tab:blue", alpha=0.15, linewidth=0, label="generic: mean ± std"
)
axs[1].plot(x, mean, "o-", color="tab:blue", label="generic: mean")
axs[1].plot(x, median, "s--", color="tab:blue", label="generic: median")
s2 = summaries["R2cut"]
axs[1].plot(x, s2["mean n_eff"].values, "o-", color="tab:red", lw=0.8, ms=3, label="R2cut: mean")
axs[1].plot(x, s2["median n_eff"].values, "s--", color="tab:red", lw=0.8, ms=3, label="R2cut: median")
axs[1].plot(x[i_ref], mean[i_ref], "o", mfc="none", mec="k", ms=12, label="baseline")
axs[1].set_xlabel("E(B-V) threshold that defines the footprint of the run")
axs[1].set_ylabel("n_eff [arcmin$^{-2}$]")
axs[1].grid(alpha=0.3)
axs[1].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "NEFF_totals_vs_dust_threshold")
plt.show()

## 11. What the visit-count proxy misses

Correlation, pixel by pixel, between `n_eff` (variant `generic`) and (i) the `WeakLensingNvisits` (`gri`) of notebook 03, (ii) the coadded `i`-band depth, (iii) the coadded `i`-band effective seeing. The scatter plot is drawn for the reference run. A weak correlation with the number of visits and a strong one with the depth and the seeing show what the seeing-aware model adds to the existing proxy.

In [ ]:
rows = []
for run_name in RUN_NAMES:
    inp = inputs[run_name]
    df = pd.DataFrame(
        {
            "n_eff": np.ma.filled(neff_maps["generic"][run_name], np.nan),
            "n_eff_R2cut": np.ma.filled(neff_maps["R2cut"][run_name], np.nan),
            "NV_gri": np.ma.filled(inp["nv"], np.nan),
            "m5_i": inp["m5"]["i"],
            "fwhm_i": inp["fwhm"]["i"],
        }
    ).dropna()
    rows.append(
        {
            "run": run_name,
            "spearman(n_eff, NV_gri)": df["n_eff"].corr(df["NV_gri"], method="spearman"),
            "spearman(n_eff, m5_i)": df["n_eff"].corr(df["m5_i"], method="spearman"),
            "spearman(n_eff, fwhm_i)": df["n_eff"].corr(df["fwhm_i"], method="spearman"),
            "spearman(n_eff_R2cut, fwhm_i)": df["n_eff_R2cut"].corr(df["fwhm_i"], method="spearman"),
        }
    )
corr = pd.DataFrame(rows).set_index("run")
corr.to_csv(join(data_dir, "neff_correlations.csv"))
display(corr.round(2))

inp = inputs[REF_RUN]
df = pd.DataFrame(
    {
        "n_eff": np.ma.filled(neff_maps["generic"][REF_RUN], np.nan),
        "NV_gri": np.ma.filled(inp["nv"], np.nan),
        "fwhm_i": inp["fwhm"]["i"],
    }
).dropna()
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
h = axs[0].hexbin(df["NV_gri"], df["n_eff"], gridsize=50, bins="log", cmap="viridis")
axs[0].set_xlabel("WeakLensingNvisits (gri)")
axs[0].set_ylabel("n_eff [arcmin$^{-2}$]")
fig.colorbar(h, ax=axs[0], label="pixels (log)")
h = axs[1].hexbin(df["fwhm_i"], df["n_eff"], gridsize=50, bins="log", cmap="viridis")
axs[1].set_xlabel("coadd effective seeing in i [arcsec]")
axs[1].set_ylabel("n_eff [arcmin$^{-2}$]")
fig.colorbar(h, ax=axs[1], label="pixels (log)")
fig.suptitle(f"{REF_RUN}: pixel-by-pixel relation of n_eff to the visits and to the seeing")
fig.tight_layout()
save_fig(fig, "NEFF_vs_NV_and_seeing_" + REF_RUN.replace(".", "_"))
plt.show()

## 12. Towards the combined metric

What is in place: a per-pixel, seeing- and depth-aware `n_eff`; the footprint quality is inherited from the WL footprint of notebook 03; `sqrt(N_eff)` relative to the baseline is a statistical signal-to-noise proxy (shape-noise dominated regime).

What is still missing for the metric described in the project:
1. **The PSF systematics term.** A term proportional to `R2_PSF / R2_gal` times the PSF-model error, decreasing with the number of well-dithered visits (Paulin-Henriksson et al. 2008; the visit-count proxy of notebook 03 is the current stand-in), so that the total noise is `sigma_SN^2 / n_eff + C_sys`. It needs a normalization from LSSTCam data.
2. **The angular power spectrum.** The noise term `sigma_SN^2 / n_eff` and the area of the footprint enter the uncertainty on the shear power spectrum (Chang et al. 2013, eq. 11) through `f_sky` and `C_l`; this requires a fiducial cosmology and redshift distribution.
3. **The absolute calibration** of `n_eff`, algorithm dependent (notebook 04).

## Caveats

- The depth and seeing maps are for the first 10 years of each run and for the visits of the bands `BANDS` only; the model does not use the `g` and `z` bands, although the WL footprint is defined from `gri` visits.
- The seeing of a pixel is the root mean square of the per-visit `FWHM` (unweighted), not the optimal-coadd PSF; the difference is small for the visit distributions of these runs but it was not checked here.
- The `n_eff` model inherits all the assumptions and the modelling uncertainty of notebook 04 (galaxy population, colours, depth definition, algorithm-dependent noise law). The differences between runs are more robust than the absolute values.
- Photometric-redshift selection, tomography, masking of bright stars and the deblending of real catalogues are not modelled.

## References
- Chang, C. et al. 2013, MNRAS 434, 2121, arXiv:1305.0793 (erratum: MNRAS 447, 1746, 2015)
- Paulin-Henriksson, S. et al. 2008, A&A 484, 67
- Lochner, M. et al. 2021, arXiv:2104.05676 (MAF weak-lensing proxy and 3x2pt emulator)
- `04_NeffSeeingModel.ipynb` (model and calibration), `03_WeakLensing.ipynb` (footprint), `01_compareExgalM5withCuts.ipynb` (depth)